In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
nltk.download('all')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_ru.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_rus to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |  

In [ ]:
text = """TextRank is an unsupervised algorithm for text summarization.
It builds a graph of sentences where edges represent similarity.
Then it applies the PageRank algorithm to score sentences.
The top-ranked sentences form the summary."""

sentences = text.split('.')
sentences

['TextRank is an unsupervised algorithm for text summarization',
 '\nIt builds a graph of sentences where edges represent similarity',
 '\nThen it applies the PageRank algorithm to score sentences',
 '\nThe top-ranked sentences form the summary',
 '']

In [ ]:
def preprocess(sentences):
  temp = []

  stop_words_set = nltk.corpus.stopwords.words('english')
  lemmatizer = WordNetLemmatizer()
  for sentence in sentences:
    if sentence != '':
      t = sentence.lower()
      t = re.sub(r'[^A-z\s]', '', t).split()
      t = [lemmatizer.lemmatize(word) for word in t if word not in stop_words_set]
      temp.append(' '.join(t))
  return temp

sentences_preprocess = preprocess(sentences)
sentences_preprocess

['textrank unsupervised algorithm text summarization',
 'build graph sentence edge represent similarity',
 'applies pagerank algorithm score sentence',
 'topranked sentence form summary']

In [ ]:
vectorizer = TfidfVectorizer(max_features=20)
document = vectorizer.fit_transform(sentences_preprocess)

document = document.toarray()

In [ ]:
document

array([[0.36673901, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.46516193, 0.        , 0.46516193, 0.46516193,
        0.        , 0.46516193],
       [0.        , 0.        , 0.43003652, 0.43003652, 0.        ,
        0.43003652, 0.        , 0.43003652, 0.        , 0.27448674,
        0.43003652, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        ],
       [0.39278432, 0.49819711, 0.        , 0.        , 0.        ,
        0.        , 0.49819711, 0.        , 0.49819711, 0.31799276,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.5417361 ,
        0.        , 0.        , 0.        , 0.        , 0.34578314,
        0.        , 0.        , 0.5417361 , 0.        , 0.        ,
        0.5417361 , 0.        ]])

In [ ]:
def compute_similarity_matrix(document):
  document = document / np.linalg.norm(document, axis=1, keepdims=True)
  sim_matrix = np.dot(document, document.T)
  return sim_matrix

sim_matrix = compute_similarity_matrix(document)
sim_matrix

array([[1.        , 0.        , 0.14404933, 0.        ],
       [0.        , 1.        , 0.0872848 , 0.09491289],
       [0.14404933, 0.0872848 , 1.        , 0.10995653],
       [0.        , 0.09491289, 0.10995653, 1.        ]])

In [ ]:
from copy import deepcopy

In [ ]:
def compute_sentence_rank(sim_matrix, damping=0.85, max_iter=100, tol=1e-4):
  n = sim_matrix.shape[0]
  ranks = np.array([1/n] * n)
  prob_matrix = sim_matrix / sim_matrix.sum(axis=1, keepdims=True)
  print(prob_matrix)

  for iter in range(max_iter):
    old_rank = deepcopy(ranks)
    ranks = damping * np.dot(prob_matrix.T, old_rank)  + (1 - damping)/n
    delta = np.sum(np.abs(ranks - old_rank))
    # print(f' Iteration = {iter} rank = {ranks} old rank = {old_rank}')
    if delta < tol:
      break

  return ranks

In [ ]:
ranks = compute_sentence_rank(sim_matrix)
ranks

[[0.87408818 0.         0.12591182 0.        ]
 [0.         0.84588222 0.07383266 0.08028512]
 [0.10739606 0.06507523 0.74555056 0.08197816]
 [0.         0.07877442 0.09126013 0.82996546]]


array([0.24081687, 0.24492946, 0.26704251, 0.24721116])

In [ ]:
idx_sorted = np.argsort(ranks)[::-1]
idx_sorted

array([2, 3, 1, 0])

In [ ]:
k = 2
sentences_summary = [sentences[i] for i in idx_sorted[:k]]
re.sub('\n','','.'.join(sentences_summary))

'Then it applies the PageRank algorithm to score sentences.The top-ranked sentences form the summary'